In [13]:
import os
import requests
from bs4 import BeautifulSoup
from datetime import date
import pandas as pd
import json
import shutil
from datetime import datetime
from google.colab import drive

# Montar o Google Drive em um novo diretório
drive.mount('/content/drive2')

# Criar uma pasta chamada "output" se não existir
output_folder = "output"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"\033[92mA pasta {output_folder} foi criada.\033[0m")

# Função para obter os dados dos produtos
def get_product_data(url, headers, page_number):
    try:
        products_data = []
        response = requests.get(f"{url}/page/{page_number}/", headers=headers)

        if response.status_code == 404:
            print(f"\033[91mPágina {page_number} não encontrada (Erro 404).\033[0m")
            return None, False

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        data_de_coleta = date.today()
        nome_empresa = "Fogões Shop"

        product_list = soup.find_all("li", class_="product-layout")

        if not product_list:
            print(f"Página {page_number} não contém produtos.")
            return None, False

        for product in product_list:
            image_tag = product.find("img", class_="img-responsive")
            image = image_tag["src"] if image_tag else ""
            name_tag = product.find("div", class_="item-title").a
            name = name_tag.text.strip() if name_tag else ""
            price_tag = product.find("span", class_="price")
            price = price_tag.text.strip().replace("R$", "").replace(".", "").replace(",", ".") if price_tag else ""
            currency = "BRL"
            availability = "Disponível"
            product_url = name_tag["href"] if name_tag else ""

            product_data = {
                "Nome empresa": nome_empresa,
                "Data de coleta": data_de_coleta,
                "Imagem": image,
                "Nome": name,
                "Preço": float(price) if price else None,
                "Moeda": currency,
                "Disponibilidade": availability,
                "URL do produto": product_url
            }
            products_data.append(product_data)

        pagination = soup.find("ul", class_="pagination")
        if pagination:
            next_page_link = pagination.find("li", text=">")
            if next_page_link:
                next_page_link = next_page_link.find("a")
            has_next_page = bool(next_page_link)
        else:
            has_next_page = False

        return products_data, has_next_page
    except requests.RequestException as e:
        print(f"\033[91mErro ao obter dados do produto na página {page_number}: {e}\033[0m")
        return None, False

def analyze_xml_links(xml_links, headers):
    for category, link in xml_links.items():
        print("Categoria:", category)
        print("Link:", link)
        product_data = []
        page_number = 1
        while True:
            print(f"Procurando produtos na página {page_number}...")
            data, has_next_page = get_product_data(link, headers, page_number)
            if not data:
                print(f"Parando a busca, nenhuma informação encontrada na página {page_number}.")
                break
            product_data.extend(data)
            if not has_next_page:
                print(f"Última página alcançada: {page_number}.")
                break
            page_number += 1

        if product_data:
            try:
                product_df = pd.DataFrame(product_data)
                # Salvar o DataFrame em um arquivo CSV dentro da pasta "output"
                product_df.to_csv(os.path.join(output_folder, f"{category}.csv"), index=False)
                print("Dados do produto salvos em", f"{category}.csv")
            except Exception as e:
                print(f"\033[91mErro ao salvar dados do produto em CSV: {e}\033[0m")
        else:
            print("\033[91mNenhum produto encontrado.\033[0m")
        print()

# Links para as categorias
links = {
    "Acessorios-banheiro": "https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-banheiro/acessorio-cabide/",
    # Adicione mais categorias conforme necessário
}

# Cabeçalhos para simular um navegador
headers = {'User-Agent': 'Mozilla/5.0'}

# Chamar a função para analisar os links
analyze_xml_links(links, headers)

# Definir o caminho do diretório base de destino no Google Drive
base_destination_dir = '/content/drive2/MyDrive/WebScraping/FogoesShop/WebScraping_coletaFogoesShop'

# Criar o caminho do novo diretório com a data de hoje
today = date.today().strftime('%Y-%m-%d')
destination_dir = os.path.join(base_destination_dir, today)

# Verificar se o diretório já existe
if not os.path.exists(destination_dir):
    os.makedirs(destination_dir, exist_ok=True)
    # Caminho do diretório de origem
    source_dir = '/content/output'

    # Mover todos os arquivos do diretório de origem para o diretório de destino
    for file_name in os.listdir(source_dir):
        full_file_name = os.path.join(source_dir, file_name)
        if os.path.isfile(full_file_name):
            shutil.move(full_file_name, destination_dir)

    # Mensagem de sucesso em verde
    print(f"\033[92mARQUIVO CRIADO {today}\033[0m")
    print(f"\033[92mARQUIVOS COLETADOS FORAM ENVIADOS PARA O GOOGLE DRIVE COM SUCESSO!\033[0m")
else:
    print(f"\033[91mDiretório para {today} já existe. Nenhum arquivo será movido.\033[0m")


Drive already mounted at /content/drive2; to attempt to forcibly remount, call drive.mount("/content/drive2", force_remount=True).
Categoria: Acessorios-banheiro
Link: https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-banheiro/acessorio-cabide/
Procurando produtos na página 1...


<ipython-input-13-c6d1c6a7af19>:67: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  next_page_link = pagination.find("li", text=">")


Procurando produtos na página 2...
Procurando produtos na página 3...
Última página alcançada: 3.
Dados do produto salvos em Acessorios-banheiro.csv

ARQUIVOS COLETADOS FORAM ENVIADOS PARA O GOOGLE DRIVE COM SUCESSO!
